In [ ]:
import json

def chunk_document(input_file, output_file):
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        if len(data) == 0:
            raise ValueError("không tìm thấy file")
        data = data[0]

    chunks = []

    # 1. Chunk cho document_info
    doc_info = data.get("document_info", {})
    if doc_info:
        info_text = []
        for k, v in doc_info.items():
            info_text.append(f"{k}: {v}")
        chunks.append({
            "article_number": None,
            "article_title": "Thông tin văn bản",
            "clause_number": None,
            "chunk_content": "\n".join(info_text)
        })

    # 2. Chunk cho preamble
    preamble = data.get("preamble", [])
    if preamble:
        chunks.append({
            "article_number": None,
            "article_title": "Căn cứ pháp lý",
            "clause_number": None,
            "chunk_content": "\n".join(preamble)
        })

    # 3. Chunks cho articles
    articles = data.get("articles", [])
    if not articles:
        print("Không tìm thấy articles")

    for article in articles:
        article_number = article.get("article_number")
        article_title = article.get("article_title")

        for clause in article.get("clauses", []):
            clause_number = clause.get("clause_number")
            content = clause.get("content", "")

            points = clause.get("points", [])
            if points:
                for p in points:
                    content += "\n\n" + p.get("content", "")

            chunks.append({
                "article_number": article_number,
                "article_title": article_title,
                "clause_number": clause_number,
                "chunk_content": content.strip()
            })

    # Xuất ra file
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    print(f"Đã chia {len(chunks)} chunks và lưu vào {output_file}")


# Ví dụ chạy
if __name__ == "__main__":
    chunk_document("Test.json", "chunks.json")


✅ Đã chia thành 220 chunks và lưu vào chunks.json


In [ ]:
import json
from sentence_transformers import SentenceTransformer

def create_embeddings(input_file, output_file, model_name="all-MiniLM-L6-v2"):
    # Load model embedding
    print(f"Đang tải model embedding: {model_name}")
    model = SentenceTransformer(model_name)

    # Đọc dữ liệu chunk
    with open(input_file, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    # Tạo embedding cho từng chunk
    for chunk in chunks:
        text = chunk.get("chunk_content", "")
        embedding = model.encode(text).tolist()  # Chuyển numpy array -> list để lưu JSON
        chunk["embedding"] = embedding

    # Lưu kết quả ra file mới
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    print(f"Đã tạo embedding cho {len(chunks)} chunks và lưu vào {output_file}")


# Ví dụ chạy
if __name__ == "__main__":
    create_embeddings("chunks.json", "chunks_with_embeddings.json")


e:\EduRAGBot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔄 Đang tải model embedding: all-MiniLM-L6-v2
✅ Đã tạo embedding cho 220 chunks và lưu vào chunks_with_embeddings.json


In [ ]:
import json
import numpy as np
import faiss
import pickle

def build_faiss_index(input_file, faiss_index_file="index.faiss", metadata_file="index.pkl"):
    # Đọc file chunks có embeddings
    with open(input_file, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    # Lấy danh sách embeddings và metadata
    embeddings = [chunk["embedding"] for chunk in chunks]
    metadata = [
        {
            "article_number": chunk.get("article_number"),
            "article_title": chunk.get("article_title"),
            "clause_number": chunk.get("clause_number"),
            "chunk_content": chunk.get("chunk_content"),
        }
        for chunk in chunks
    ]

    # Convert embeddings -> numpy array (float32)
    embeddings = np.array(embeddings).astype("float32")

    # Khởi tạo FAISS index (dùng cosine similarity ~ inner product)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    faiss.normalize_L2(embeddings)

    # Thêm embeddings vào index
    index.add(embeddings)

    # Lưu FAISS index
    faiss.write_index(index, faiss_index_file)

    # Lưu metadata bằng pickle
    with open(metadata_file, "wb") as f:
        pickle.dump(metadata, f)

    print(f"Đã build FAISS index với {len(metadata)} vectors")
    print(f"   FAISS: {faiss_index_file}")
    print(f"   Metadata pickle: {metadata_file}")

    # Ví dụ chạy
if __name__ == "__main__":
    build_faiss_index(
        "chunks_with_embeddings.json",
        )


✅ Đã build FAISS index với 220 vectors
   FAISS lưu ở: index.faiss
   Metadata pickle lưu ở: index.pkl


In [13]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

# ===== Load index + metadata =====
def load_faiss_index(faiss_index_file="index.faiss", metadata_file="index.pkl"):
    # Load FAISS index
    index = faiss.read_index(faiss_index_file)

    # Load metadata
    with open(metadata_file, "rb") as f:
        metadata = pickle.load(f)

    return index, metadata


# ===== Search trong FAISS =====
def search_faiss(query, index, metadata, model, top_k=5):
    # Encode query thành vector
    query_vec = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_vec)

    # Tìm top_k vectors gần nhất
    distances, indices = index.search(query_vec, top_k)

    results = []
    for i, idx in enumerate(indices[0]):
        if idx == -1:  # Nếu không tìm thấy
            continue
        results.append({
            "rank": i+1,
            "score": float(distances[0][i]),
            "metadata": metadata[idx]
        })

    return results


# ===== Ví dụ chạy =====
if __name__ == "__main__":
    # Load lại index + metadata
    index, metadata = load_faiss_index("index.faiss", "index.pkl")

    # Dùng đúng model đã build index
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Query thử
    query = "Đào tạo văn bằng thứ hai"
    results = search_faiss(query, index, metadata, model, top_k=3)

    # In kết quả
    for r in results:
        print(f"Rank {r['rank']} | Score: {r['score']:.4f}")
        print(f"Article {r['metadata'].get('article_number')} - {r['metadata'].get('article_title')}")
        print(f"Content: {r['metadata']['chunk_content'][:200]}...\n")


Rank 1 | Score: 0.8081
Article 25 - Đào tạo văn bằng thứ hai
Content: Đào tạo văn bằng thứ hai theo hình thức vừa làm vừa học dành cho người đã có bằng đại học....

Rank 2 | Score: 0.7004
Article 12 - Nguyên tắc xây dựng ngành học mới
Content: Đơn vị đào tạo xây dựng đề án mở ngành học mới theo nguyên tắc:...

Rank 3 | Score: 0.6966
Article 21 - Tổ chức đăng ký học phần
Content: Đối với học kỳ phụ: Thủ trưởng đơn vị đào tạo quy định....

